# GenZ Bible RAG Agent 🔥📖

In [1]:
from setup import bedrock_tool
import os
import chromadb
from chromadb.utils import embedding_functions
from agents import Agent, Runner, function_tool, trace
from agents.mcp import MCPServerStreamableHttp

✅ AWS credentials found
✅ OpenAI credentials found
✅ EXA credentials found


## Connect to ChromaDB
We populated the RAG with GenZ Bible data in `../rag_setup/rag_setup.ipynb`.
We must use the same embedding model here as when we created the collection.

In [2]:
# Must match the embedding model used in rag_setup.ipynb
ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/all-MiniLM-L12-v2"
)

chroma_client = chromadb.PersistentClient("../chroma")
bible_db = chroma_client.get_collection(name="genz_bible", embedding_function=ef)

print(f"✅ Connected to collection with {bible_db.count()} verses")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Connected to collection with 21 verses


## Quick sanity check

In [3]:
results = bible_db.query(query_texts=["God so loved the world"], n_results=2)
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(f"📖 {meta['reference']}")
    print(f"   KJV:  {doc[:100]}")
    print(f"   GenZ: {meta.get('genz_text', 'N/A')[:100]}")
    print()

📖 John 13
   KJV:  So, right before the feast of the passover, Jesus knew it was time to peace out of this world and go
   GenZ: N/A

📖 John 1
   KJV:  So like, when everything first started, there was the Word . And the Word was with God , and the Wor
   GenZ: N/A



## Define the RAG tool

In [4]:
@function_tool
def bible_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Look up relevant Bible verses for a given question or topic.
    Searches using the original KJV text for accuracy, and returns
    the GenZ translation for the answer.

    Use this tool FIRST before falling back to Exa web search.
    If this tool returns 'NO_RELEVANT_RESULTS', use Exa to search
    the web for the answer and translate it into GenZ language yourself.

    Args:
        query: The question or topic to look up (e.g. 'love your enemies', 'creation of the world').
        max_results: The maximum number of verses to return.

    Returns:
        A string containing the relevant Bible verses in GenZ translation,
        or 'NO_RELEVANT_RESULTS' if nothing relevant was found.
    """
    results = bible_db.query(
        query_texts=[query],
        n_results=max_results,
        include=["documents", "metadatas", "distances"]
    )

    if not results["documents"][0]:
        return "NO_RELEVANT_RESULTS"

    # Filter out low-relevance results using cosine distance threshold
    # Distance > 1.0 means the result is likely not relevant
    RELEVANCE_THRESHOLD = 1.0
    formatted_results = []
    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        if distance > RELEVANCE_THRESHOLD:
            continue
        reference = meta["reference"]
        genz_text = meta.get("genz_text", doc)  # fall back to KJV if genz_text missing
        formatted_results.append(f"{reference}: {genz_text}")

    if not formatted_results:
        return "NO_RELEVANT_RESULTS"

    return "Relevant Bible verses (GenZ translation):\n" + "\n".join(formatted_results)

In [5]:
# Test the tool before wiring it into the agent
# bible_lookup_tool('what did Jesus say about love')

## Define the Agent

In [5]:
# Set up Exa MCP — used as fallback when RAG has no relevant results
exa_search_mcp = MCPServerStreamableHttp(
    name="Exa Search MCP",
    params={
        "url": f"https://mcp.exa.ai/mcp?exaApiKey={os.environ.get('EXA_API_KEY')}",
        "timeout": 90,
    },
    client_session_timeout_seconds=90,
    cache_tools_list=True,
    max_retry_attempts=1,
)

await exa_search_mcp.connect()

bible_agent = Agent(
    name="GenZ Bible Assistant",
    instructions="""
    You are a Bible assistant who answers questions about the Bible using GenZ language.
    You are knowledgeable, engaging, and keep it real with the user. No cap.

    Follow this workflow for every question:
    1) ALWAYS try bible_lookup_tool first to find relevant verses from our database.
    2) If bible_lookup_tool returns 'NO_RELEVANT_RESULTS', fall back to Exa web search
       to find the answer from a trusted GenZ Bible website (e.g. https://genz.bible/).
       Then use the GenZ language translation when responding.
    3) Never skip step 1 — always check the RAG database before going to the web.

    When answering:
    - Answer using the GenZ translation from the tool, or the GenZ translation from genz bible if using Exa
    - Keep answers concise but insightful
    - Use GenZ slang naturally (e.g. fr fr, no cap, bussin, slay, lowkey, vibe, it's giving)
    - Always cite the Bible reference (e.g. John 3:16) when quoting a verse
    - If the question is not related to the Bible at all, let the user know that's not your vibe
    """,
    model="litellm/bedrock/eu.amazon.nova-lite-v1:0",
    tools=[bedrock_tool(bible_lookup_tool.__dict__)],  # RAG tool
    mcp_servers=[exa_search_mcp],                      # Exa fallback
)

## Run the Agent

In [6]:
# ── Test 1: Should hit RAG ──────────────────────────────────────
with trace("GenZ Bible Assistant - RAG hit") as t:
    result = await Runner.run(
        bible_agent,
        "What did Jesus say about loving your enemies?",
    )
print(f"Output: {result.final_output}")
print(f"Trace: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

Output: No cap, here's what Jesus had to say about loving your enemies, straight from the Bible (GenZ translation):

"But I tell you, love your enemies and pray for those who persecute you, so that you may be children of your Father in heaven. He makes his sun rise on the evil and the good, and sends rain on the righteous and the unrighteous alike." - Matthew 5:44-45

And another one from Luke:

"But I tell you who will listen: Love your enemies, do good to those who hate you, bless those who curse you, pray for those who mistreat you." - Luke 6:27-28

It's wild, right? Jesus was all about spreading love and peace, even to those who might not deserve it.
Trace: https://platform.openai.com/logs/trace?trace_id=trace_7286b392bb014fc19d54d106e5ee039f


In [7]:
# ── Test 2: Should hit RAG ──────────────────────────────────────
with trace("GenZ Bible Assistant - RAG hit") as t:
    result = await Runner.run(
        bible_agent,
        "What's the tea on the creation of the world?",
    )
print(f"Output: {result.final_output}")
print(f"Trace: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

Output: So, here's the deets on the creation of the world according to the Bible:

In the beginning, there was this thing called the Word, and it was all about God, no cap (John 1:1). And then, after a lot went down, Jesus showed up again to his peeps by the sea of Tiberias, just to remind them he's still in the game (John 21:1). But before all that, Jesus knew it was time to leave this world and head back to the Father, showing mad love for his squad (John 13:1).
Trace: https://platform.openai.com/logs/trace?trace_id=trace_d3c98d445f5a4a98a10e333633ea284e


In [8]:
# ── Test 3: Should trigger Exa fallback ────────────────────────
# This asks about a book we likely didn't scrape — Exa should kick in
with trace("GenZ Bible Assistant - Exa fallback") as t:
    result = await Runner.run(
        bible_agent,
        "What happened in the book of Revelation with the seven seals?",
    )
print(f"Output: {result.final_output}")
print(f"Trace: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

Output: Alright, so here's the lowdown on the seven seals in the book of Revelation, GenZ style:

When John saw the Lamb (Jesus) take the scroll and try to open the seven seals, nothing in heaven, earth, or under the earth could open it or even peep at it (Revelation 5:1-4). 

Then, one of the elders tells John that the Lion of Judah, the Root of David, has prevailed and is worthy to open the scroll and its seven seals (Revelation 5:5). When the Lamb takes the scroll from the right hand of the one on the throne, the four living creatures and the twenty-four elders fall down before the Lamb, each having harps and golden bowls full of incense, which are the prayers of the saints (Revelation 5:8-9).

When the Lamb opens the first four seals, it unleashes a series of judgments on the earth, including conquest, violence, scarcity, and death (Revelation 6:1-8). The opening of the fifth seal reveals the souls of martyrs crying out for justice, while the sixth seal brings about cosmic upheaval